# DeepVerse-Python Digital Twin Beam Prediction Demo

## Introduction

In this notebook, we present a digital twin beam prediciton demo of the [DeepVerse 6G](https://deepverse6g.net/) dataset.
- We train a position-based beam prediction NN on the digital twin (synthetic) data.
- The trained model is tested on the corresponding real-world data.

See more detailed [API documentation](https://deepverse6g.net/documentation)\
See more detailed [DeepVerse generator tutorial](https://deepverse6g.net/tutorials)

Please cite the following paper if you use this notebook (parts or modified version) in your research project.

In [1]:
'''
@inproceedings{jiang2023digital,
  title={Digital twin based beam prediction: Can we train in the digital world and deploy in reality?},
  author={Jiang, Shuaifeng and Alkhateeb, Ahmed},
  booktitle={2023 IEEE International Conference on Communications Workshops (ICC Workshops)},
  pages={36--41},
  year={2023},
  organization={IEEE}
}
'''

'\n@inproceedings{jiang2023digital,\n  title={Digital twin based beam prediction: Can we train in the digital world and deploy in reality?},\n  author={Jiang, Shuaifeng and Alkhateeb, Ahmed},\n  booktitle={2023 IEEE International Conference on Communications Workshops (ICC Workshops)},\n  pages={36--41},\n  year={2023},\n  organization={IEEE}\n}\n'

## Installation

In [2]:
# Installation needed starting from a fresh conda env
# This may not produce a torch installation supporting cuda and gpu
# %pip install torch numpy pandas tqdm scikit-learn requests deepverse scipy

## Import

In [6]:
import os
import sys
from scipy.io import loadmat
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset as TorchDataset, DataLoader
from deepverse import ParameterManager, Dataset as DeepVerseDataset

In [7]:
import os, glob
import pandas as pd
import numpy as np

base = "scenarios/DT31"

print("=== param folder ===")
print(os.listdir(f"{base}/param"))

print("\n=== csv files in scenario root ===")
print(glob.glob(f"{base}/*.csv"))

print("\n=== mat files in param ===")
print(glob.glob(f"{base}/param/*.mat"))

=== param folder ===
['trajectory.mat', 'config.m', 'scenario.yaml']

=== csv files in scenario root ===
[]

=== mat files in param ===
['scenarios/DT31/param/trajectory.mat']


In [10]:
from scipy.io import loadmat

mat_path = "scenarios/DT31/param/trajectory.mat"
traj = loadmat(mat_path)

print(traj.keys())

dict_keys(['__header__', '__version__', '__globals__', 'object_info', 'scene'])


In [11]:
import numpy as np

for k in ["object_info", "scene"]:
    v = traj[k]
    print(f"\n=== {k} ===")
    print("type:", type(v))
    print("shape:", np.array(v).shape)
    print("dtype:", np.array(v).dtype)
    print("value preview:")
    print(v)


=== object_info ===
type: <class 'numpy.ndarray'>
shape: (1, 1)
dtype: object
value preview:
[[array([[(array(['user'], dtype='<U4'), array([[4.22]]), array([[2.]]), array([[1.48]]), array(['sedan'], dtype='<U5'))]],
        dtype=[('id', 'O'), ('length', 'O'), ('width', 'O'), ('height', 'O'), ('vClass', 'O')])                            ]]

=== scene ===
type: <class 'numpy.ndarray'>
shape: (1, 7012)
dtype: object
value preview:
[[array([[(array([[0.]]), array([[array([[(array([[0]], dtype=int32), array([[-44.69214507]]), array([[94.32385129]]), array([[0]], dtype=int32), array([[4.47]]), array([[-86.]]), array([[0]], dtype=int32), array(['user'], dtype='<U4'), array([[1.68]]), array([[0]], dtype=int32), array([[-4.68521451e+01,  9.32738513e+01, -5.00000000e-02],
                                 [-4.25321451e+01,  9.53738513e+01,  1.73000000e+00]]))                                                                                                                                         

# DeepVerse-Python Digital Twin Beam Prediction Demo

## Introduction

In this notebook, we present a digital twin beam prediciton demo of the [DeepVerse 6G](https://deepverse6g.net/) dataset.
- We train a position-based beam prediction NN on the digital twin (synthetic) data.
- The trained model is tested on the corresponding real-world data.

See more detailed [API documentation](https://deepverse6g.net/documentation)\
See more detailed [DeepVerse generator tutorial](https://deepverse6g.net/tutorials)

Please cite the following paper if you use this notebook (parts or modified version) in your research project.

In [ ]:
'''
@inproceedings{jiang2023digital,
  title={Digital twin based beam prediction: Can we train in the digital world and deploy in reality?},
  author={Jiang, Shuaifeng and Alkhateeb, Ahmed},
  booktitle={2023 IEEE International Conference on Communications Workshops (ICC Workshops)},
  pages={36--41},
  year={2023},
  organization={IEEE}
}
'''

'\n@inproceedings{jiang2023digital,\n  title={Digital twin based beam prediction: Can we train in the digital world and deploy in reality?},\n  author={Jiang, Shuaifeng and Alkhateeb, Ahmed},\n  booktitle={2023 IEEE International Conference on Communications Workshops (ICC Workshops)},\n  pages={36--41},\n  year={2023},\n  organization={IEEE}\n}\n'

## Installation

In [ ]:
# Installation needed starting from a fresh conda env
# This may not produce a torch installation supporting cuda and gpu
# %pip install torch numpy pandas tqdm scikit-learn requests deepverse scipy

## Import

In [ ]:
import os
import sys
from scipy.io import loadmat
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset as TorchDataset, DataLoader
from deepverse import ParameterManager, Dataset as DeepVerseDataset

## Digital twin synthetic data preparation

### Download DeepVerse data

Here we download the DeepVerse (synthetic) data. We use the DT1 scenario, which incoporates the LiDAR, RGB images, wireless, and position data modalities. Since this demo focuses on the position-based beam prediction application, we only download and use the wireless data modality.

The scenario folder follows the below structure
```
DeepVerse-main/
├─ scenarios/
│  ├─ DT1/
│  │  ├─ wireless/
│  │  │  ├─ ...
│  │  │  ├─ params.mat
│  │  ├─ param/
│  │  │  ├─ config.m
│  │  ├─ scenario1.csv
```

### DeepVerse dataset generation

In [ ]:
scenario_name = "DT31"
config_path = f"scenarios/{scenario_name}/param/config.m"

param_manager = ParameterManager(config_path)

# 여기 scene 수는 네가 실제 쓸 값으로 하나만 정해
param_manager.params["scenes"] = list(range(100))

param_manager.params["radar"]["enable"] = False
param_manager.params["comm"]["enable"] = True
param_manager.params["camera"] = False
param_manager.params["lidar"] = False
param_manager.params["position"] = True

dataset = DeepVerseDataset(param_manager)

Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (0.83s)


### Data postprcessing

The generated DeepVerse dataset consists of:
- The communication channel between the basestation and the user
- The position of the communication user

In the following blocks we apply postprocessing on this data:
- We apply beamforming (from a beam codebook) to the communication channel to get the communication beam power
- We will then save the postprocessed data: beam power and user position.

#### Simple beam-steering codebook

In [ ]:
def beam_steering_codebook(angles, num_z, num_x):
    d = 0.5
    k_z = np.arange(num_z)
    k_x = np.arange(num_x)
    
    codebook = []
    
    for beam_idx in range(angles.shape[0]):
        z_angle = angles[beam_idx, 0]
        x_angle = angles[beam_idx, 1]
        bf_vector_z = np.exp(1j * 2 * np.pi * k_z * d * np.cos(np.radians(z_angle)))
        bf_vector_x = np.exp(1j * 2 * np.pi * k_x * d * np.cos(np.radians(x_angle)))
        bf_vector = np.outer(bf_vector_z, bf_vector_x).flatten()
        codebook.append(bf_vector)
        
    return np.stack(codebook, axis=0)

# construct beam steering codebook
x_angles = loadmat(f'scenarios/{scenario_name}/param/beam_angles.mat')['beam_angles'].squeeze() * 180 / np.pi
num_angles = x_angles.size

# x_angles = np.flip(x_angles)
z_angles = np.full(num_angles, 90)
beam_angles = np.column_stack((z_angles, x_angles))
codebook =  beam_steering_codebook(beam_angles, 1, 16)

FileNotFoundError: [Errno 2] No such file or directory: 'scenarios/DT31/param/beam_angles.mat'

#### Save communication beam power and position

In [ ]:
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# apply codebook to bs-ue comm channel
beam_power = []
position = []
num_scene = len(dataset.params['scenes'])
for i in range(num_scene):
    channel = dataset.get_sample('comm-ue', index=i, bs_idx=0, ue_idx=0).coeffs
    beam_power_ = (np.abs(codebook @ np.squeeze(channel, 0))**2).sum(-1)
    beam_power.append(beam_power_)
    position.append(dataset.get_sample('loc-ue', index=i, bs_idx=0, ue_idx=0))
np.save(f'{data_dir}/dt_beam_power.npy', np.stack(beam_power, 0))
np.save(f'{data_dir}/pos.npy', np.stack(position, 0))

In [ ]:
scene_indices = np.arange(len(position))

train_idx, test_idx = train_test_split(
    scene_indices,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

np.save(f"{data_dir}/{scenario_name}_train_idx.npy", train_idx)
np.save(f"{data_dir}/{scenario_name}_test_idx.npy", test_idx)

# DeepVerse-Python Digital Twin Beam Prediction Demo

## Introduction

In this notebook, we present a digital twin beam prediciton demo of the [DeepVerse 6G](https://deepverse6g.net/) dataset.
- We train a position-based beam prediction NN on the digital twin (synthetic) data.
- The trained model is tested on the corresponding real-world data.

See more detailed [API documentation](https://deepverse6g.net/documentation)\
See more detailed [DeepVerse generator tutorial](https://deepverse6g.net/tutorials)

Please cite the following paper if you use this notebook (parts or modified version) in your research project.

In [ ]:
'''
@inproceedings{jiang2023digital,
  title={Digital twin based beam prediction: Can we train in the digital world and deploy in reality?},
  author={Jiang, Shuaifeng and Alkhateeb, Ahmed},
  booktitle={2023 IEEE International Conference on Communications Workshops (ICC Workshops)},
  pages={36--41},
  year={2023},
  organization={IEEE}
}
'''

## Installation

In [ ]:
# Installation needed starting from a fresh conda env
# This may not produce a torch installation supporting cuda and gpu
# %pip install torch numpy pandas tqdm scikit-learn requests deepverse scipy

## Import

In [ ]:
import os
import sys
from scipy.io import loadmat
import torch
import torch.nn as nn
import zipfile
import requests
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset as TorchDataset, DataLoader
from deepverse import ParameterManager, Dataset as DeepVerseDataset

## Digital twin synthetic data preparation

### Download DeepVerse data

Here we download the DeepVerse (synthetic) data. We use the DT1 scenario, which incoporates the LiDAR, RGB images, wireless, and position data modalities. Since this demo focuses on the position-based beam prediction application, we only download and use the wireless data modality.

The scenario folder follows the below structure
```
DeepVerse-main/
├─ scenarios/
│  ├─ DT1/
│  │  ├─ wireless/
│  │  │  ├─ ...
│  │  │  ├─ params.mat
│  │  ├─ param/
│  │  │  ├─ config.m
│  │  ├─ scenario1.csv
```

In [ ]:
from pathlib import Path
from deepverse import ParameterManager, Dataset

scenario_name = "DT31"
config_path = f"scenarios/{scenario_name}/param/config.m"

param_manager = ParameterManager(config_path)
param_manager.params["scenes"] = list(range(100))   # 예시
param_manager.params["comm"]["enable"] = True
param_manager.params["position"] = True

dataset = Dataset(param_manager)

Preparing wireless data


Downloading: wireless.zip: 100%|██████████| 129M/129M [00:01<00:00, 84.4MB/s] 


Infalting: scenarios\DT1\wireless.zip
Removing zip: scenarios\DT1\wireless.zip
Preparing param files


Downloading: param.zip: 100%|██████████| 3.19M/3.19M [00:00<00:00, 13.5MB/s]

Infalting: scenarios\DT1\param.zip
Removing zip: scenarios\DT1\param.zip
Completed


### DeepVerse dataset generation

In [ ]:
scenario_name = "DT31"
config_path = f"scenarios/{scenario_name}/param/config.m"

param_manager = ParameterManager(config_path)

# 여기 scene 수는 네가 실제 쓸 값으로 하나만 정해
param_manager.params["scenes"] = list(range(100))

param_manager.params["radar"]["enable"] = False
param_manager.params["comm"]["enable"] = True
param_manager.params["camera"] = False
param_manager.params["lidar"] = False
param_manager.params["position"] = True

dataset = DeepVerseDataset(param_manager)

Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (70.26s)


### Data postprcessing

The generated DeepVerse dataset consists of:
- The communication channel between the basestation and the user
- The position of the communication user

In the following blocks we apply postprocessing on this data:
- We apply beamforming (from a beam codebook) to the communication channel to get the communication beam power
- We will then save the postprocessed data: beam power and user position.

#### Simple beam-steering codebook

In [ ]:
def beam_steering_codebook(angles, num_z, num_x):
    d = 0.5
    k_z = np.arange(num_z)
    k_x = np.arange(num_x)
    
    codebook = []
    
    for beam_idx in range(angles.shape[0]):
        z_angle = angles[beam_idx, 0]
        x_angle = angles[beam_idx, 1]
        bf_vector_z = np.exp(1j * 2 * np.pi * k_z * d * np.cos(np.radians(z_angle)))
        bf_vector_x = np.exp(1j * 2 * np.pi * k_x * d * np.cos(np.radians(x_angle)))
        bf_vector = np.outer(bf_vector_z, bf_vector_x).flatten()
        codebook.append(bf_vector)
        
    return np.stack(codebook, axis=0)

# construct beam steering codebook
x_angles = loadmat(f'scenarios/{scenario_name}/param/beam_angles.mat')['beam_angles'].squeeze() * 180 / np.pi
num_angles = x_angles.size

# x_angles = np.flip(x_angles)
z_angles = np.full(num_angles, 90)
beam_angles = np.column_stack((z_angles, x_angles))
codebook =  beam_steering_codebook(beam_angles, 1, 16)

#### Save communication beam power and position

In [ ]:
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# apply codebook to bs-ue comm channel
beam_power = []
position = []
num_scene = len(dataset.params['scenes'])
for i in range(num_scene):
    channel = dataset.get_sample('comm-ue', index=i, bs_idx=0, ue_idx=0).coeffs
    beam_power_ = (np.abs(codebook @ np.squeeze(channel, 0))**2).sum(-1)
    beam_power.append(beam_power_)
    position.append(dataset.get_sample('loc-ue', index=i, bs_idx=0, ue_idx=0))
np.save(f'{data_dir}/dt_beam_power.npy', np.stack(beam_power, 0))
np.save(f'{data_dir}/pos.npy', np.stack(position, 0))

## Digital twin synthetic data preparation

### Download DeepVerse data

Here we download the DeepVerse (synthetic) data. We use the DT1 scenario, which incoporates the LiDAR, RGB images, wireless, and position data modalities. Since this demo focuses on the position-based beam prediction application, we only download and use the wireless data modality.

The scenario folder follows the below structure
```
DeepVerse-main/
├─ scenarios/
│  ├─ DT1/
│  │  ├─ wireless/
│  │  │  ├─ ...
│  │  │  ├─ params.mat
│  │  ├─ param/
│  │  │  ├─ config.m
│  │  ├─ scenario1.csv
```

### DeepVerse dataset generation

In [4]:
scenario_name = "DT31"
config_path = f"scenarios/{scenario_name}/param/config.m"

param_manager = ParameterManager(config_path)

# 여기 scene 수는 네가 실제 쓸 값으로 하나만 정해
param_manager.params["scenes"] = list(range(100))

param_manager.params["radar"]["enable"] = False
param_manager.params["comm"]["enable"] = True
param_manager.params["camera"] = False
param_manager.params["lidar"] = False
param_manager.params["position"] = True

dataset = DeepVerseDataset(param_manager)

Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (0.83s)


### Data postprcessing

The generated DeepVerse dataset consists of:
- The communication channel between the basestation and the user
- The position of the communication user

In the following blocks we apply postprocessing on this data:
- We apply beamforming (from a beam codebook) to the communication channel to get the communication beam power
- We will then save the postprocessed data: beam power and user position.

#### Simple beam-steering codebook

In [5]:
def beam_steering_codebook(angles, num_z, num_x):
    d = 0.5
    k_z = np.arange(num_z)
    k_x = np.arange(num_x)
    
    codebook = []
    
    for beam_idx in range(angles.shape[0]):
        z_angle = angles[beam_idx, 0]
        x_angle = angles[beam_idx, 1]
        bf_vector_z = np.exp(1j * 2 * np.pi * k_z * d * np.cos(np.radians(z_angle)))
        bf_vector_x = np.exp(1j * 2 * np.pi * k_x * d * np.cos(np.radians(x_angle)))
        bf_vector = np.outer(bf_vector_z, bf_vector_x).flatten()
        codebook.append(bf_vector)
        
    return np.stack(codebook, axis=0)

# construct beam steering codebook
x_angles = loadmat(f'scenarios/{scenario_name}/param/beam_angles.mat')['beam_angles'].squeeze() * 180 / np.pi
num_angles = x_angles.size

# x_angles = np.flip(x_angles)
z_angles = np.full(num_angles, 90)
beam_angles = np.column_stack((z_angles, x_angles))
codebook =  beam_steering_codebook(beam_angles, 1, 16)

FileNotFoundError: [Errno 2] No such file or directory: 'scenarios/DT31/param/beam_angles.mat'

#### Save communication beam power and position

In [ ]:
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# apply codebook to bs-ue comm channel
beam_power = []
position = []
num_scene = len(dataset.params['scenes'])
for i in range(num_scene):
    channel = dataset.get_sample('comm-ue', index=i, bs_idx=0, ue_idx=0).coeffs
    beam_power_ = (np.abs(codebook @ np.squeeze(channel, 0))**2).sum(-1)
    beam_power.append(beam_power_)
    position.append(dataset.get_sample('loc-ue', index=i, bs_idx=0, ue_idx=0))
np.save(f'{data_dir}/dt_beam_power.npy', np.stack(beam_power, 0))
np.save(f'{data_dir}/pos.npy', np.stack(position, 0))

In [ ]:
scene_indices = np.arange(len(position))

train_idx, test_idx = train_test_split(
    scene_indices,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

np.save(f"{data_dir}/{scenario_name}_train_idx.npy", train_idx)
np.save(f"{data_dir}/{scenario_name}_test_idx.npy", test_idx)

# DeepVerse-Python Digital Twin Beam Prediction Demo

## Introduction

In this notebook, we present a digital twin beam prediciton demo of the [DeepVerse 6G](https://deepverse6g.net/) dataset.
- We train a position-based beam prediction NN on the digital twin (synthetic) data.
- The trained model is tested on the corresponding real-world data.

See more detailed [API documentation](https://deepverse6g.net/documentation)\
See more detailed [DeepVerse generator tutorial](https://deepverse6g.net/tutorials)

Please cite the following paper if you use this notebook (parts or modified version) in your research project.

In [ ]:
'''
@inproceedings{jiang2023digital,
  title={Digital twin based beam prediction: Can we train in the digital world and deploy in reality?},
  author={Jiang, Shuaifeng and Alkhateeb, Ahmed},
  booktitle={2023 IEEE International Conference on Communications Workshops (ICC Workshops)},
  pages={36--41},
  year={2023},
  organization={IEEE}
}
'''

## Installation

In [ ]:
# Installation needed starting from a fresh conda env
# This may not produce a torch installation supporting cuda and gpu
# %pip install torch numpy pandas tqdm scikit-learn requests deepverse scipy

## Import

In [ ]:
import os
import sys
from scipy.io import loadmat
import torch
import torch.nn as nn
import zipfile
import requests
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset as TorchDataset, DataLoader
from deepverse import ParameterManager, Dataset as DeepVerseDataset

## Digital twin synthetic data preparation

### Download DeepVerse data

Here we download the DeepVerse (synthetic) data. We use the DT1 scenario, which incoporates the LiDAR, RGB images, wireless, and position data modalities. Since this demo focuses on the position-based beam prediction application, we only download and use the wireless data modality.

The scenario folder follows the below structure
```
DeepVerse-main/
├─ scenarios/
│  ├─ DT1/
│  │  ├─ wireless/
│  │  │  ├─ ...
│  │  │  ├─ params.mat
│  │  ├─ param/
│  │  │  ├─ config.m
│  │  ├─ scenario1.csv
```

In [ ]:
from pathlib import Path
from deepverse import ParameterManager, Dataset

scenario_name = "DT31"
config_path = f"scenarios/{scenario_name}/param/config.m"

param_manager = ParameterManager(config_path)
param_manager.params["scenes"] = list(range(100))   # 예시
param_manager.params["comm"]["enable"] = True
param_manager.params["position"] = True

dataset = Dataset(param_manager)

Preparing wireless data


Downloading: wireless.zip: 100%|██████████| 129M/129M [00:01<00:00, 84.4MB/s] 


Infalting: scenarios\DT1\wireless.zip
Removing zip: scenarios\DT1\wireless.zip
Preparing param files


Downloading: param.zip: 100%|██████████| 3.19M/3.19M [00:00<00:00, 13.5MB/s]

Infalting: scenarios\DT1\param.zip
Removing zip: scenarios\DT1\param.zip
Completed


### DeepVerse dataset generation

In [ ]:
scenario_name = "DT31"
config_path = f"scenarios/{scenario_name}/param/config.m"

param_manager = ParameterManager(config_path)

# 여기 scene 수는 네가 실제 쓸 값으로 하나만 정해
param_manager.params["scenes"] = list(range(100))

param_manager.params["radar"]["enable"] = False
param_manager.params["comm"]["enable"] = True
param_manager.params["camera"] = False
param_manager.params["lidar"] = False
param_manager.params["position"] = True

dataset = DeepVerseDataset(param_manager)

Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (70.26s)


### Data postprcessing

The generated DeepVerse dataset consists of:
- The communication channel between the basestation and the user
- The position of the communication user

In the following blocks we apply postprocessing on this data:
- We apply beamforming (from a beam codebook) to the communication channel to get the communication beam power
- We will then save the postprocessed data: beam power and user position.

#### Simple beam-steering codebook

In [ ]:
def beam_steering_codebook(angles, num_z, num_x):
    d = 0.5
    k_z = np.arange(num_z)
    k_x = np.arange(num_x)
    
    codebook = []
    
    for beam_idx in range(angles.shape[0]):
        z_angle = angles[beam_idx, 0]
        x_angle = angles[beam_idx, 1]
        bf_vector_z = np.exp(1j * 2 * np.pi * k_z * d * np.cos(np.radians(z_angle)))
        bf_vector_x = np.exp(1j * 2 * np.pi * k_x * d * np.cos(np.radians(x_angle)))
        bf_vector = np.outer(bf_vector_z, bf_vector_x).flatten()
        codebook.append(bf_vector)
        
    return np.stack(codebook, axis=0)

# construct beam steering codebook
x_angles = loadmat(f'scenarios/{scenario_name}/param/beam_angles.mat')['beam_angles'].squeeze() * 180 / np.pi
num_angles = x_angles.size

# x_angles = np.flip(x_angles)
z_angles = np.full(num_angles, 90)
beam_angles = np.column_stack((z_angles, x_angles))
codebook =  beam_steering_codebook(beam_angles, 1, 16)

#### Save communication beam power and position

In [ ]:
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# apply codebook to bs-ue comm channel
beam_power = []
position = []
num_scene = len(dataset.params['scenes'])
for i in range(num_scene):
    channel = dataset.get_sample('comm-ue', index=i, bs_idx=0, ue_idx=0).coeffs
    beam_power_ = (np.abs(codebook @ np.squeeze(channel, 0))**2).sum(-1)
    beam_power.append(beam_power_)
    position.append(dataset.get_sample('loc-ue', index=i, bs_idx=0, ue_idx=0))
np.save(f'{data_dir}/dt_beam_power.npy', np.stack(beam_power, 0))
np.save(f'{data_dir}/pos.npy', np.stack(position, 0))